In [ ]:
!pip install python-dotenv azure-search-documents==11.6.0b7 azure-identity openai tiktoken

In [ ]:
import sqlite3
import json
import os
from openai import AzureOpenAI
import tiktoken
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# Azure OpenAI configuration
azure_openai_embedding_deployment = "text-embedding-ada-002"
azure_openai_embedding_dimensions = 1536  
azure_openai_api_version = "2023-05-15"
azure_openai_endpoint = "https://adummy-endpoint-name.openai.azure.com/"
azure_openai_key = "DummyKey"  
embedding_model_name = "text-embedding-ada-002"  
azure_search_api_key = "DummyKey"
azure_search_endpoint = "https://Dummy.search.windows.net/"


openai_credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(openai_credential, "https://cognitiveservices.azure.com/.default")


client = AzureOpenAI(
    azure_deployment=azure_openai_embedding_deployment,
    api_version=azure_openai_api_version,
    azure_endpoint=azure_openai_endpoint,
    api_key=azure_openai_key,
    azure_ad_token_provider=token_provider if not azure_openai_key else None
)

# Connect to SQLite DB
conn = sqlite3.connect("notebooks.db") 
cursor = conn.cursor()
cursor.execute("SELECT id, name, content FROM notebooks")  

records = cursor.fetchall()
conn.close()

# Prepare data
input_data = []
for row in records:
    id_, name, content = row
    try:
        description_part = content.split("Description of the notebook:")[1].split("\n")[0].strip()
        source_code_part = content.split("Source Code:")[1].strip()
    except IndexError:
        description_part = ""
        source_code_part = ""

    notebook_content = f"{description_part}\n\n{source_code_part}"
    input_data.append({
        "id": id_,
        "title": name,
        "content": notebook_content
    })
input_data[:5]

In [11]:
def chunk_text(text, max_tokens=8000, model_name="text-embedding-ada-002"):
    encoding = tiktoken.encoding_for_model(model_name)
    tokens = encoding.encode(text)
    
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i+max_tokens]
        chunks.append(encoding.decode(chunk_tokens))
    return chunks


In [12]:
import numpy as np

def embed_and_average_chunks(text, client, model_name="text-embedding-ada-002"):
    chunks = chunk_text(text)
    responses = client.embeddings.create(input=chunks, model=model_name)
    vectors = np.array([r.embedding for r in responses.data])
    return vectors.mean(axis=0).tolist()


In [13]:
notebook_vectors = []

for item in input_data:
    try:
        title_vector = embed_and_average_chunks(item["title"], client)
        content_vector = embed_and_average_chunks(item["content"], client)
        
        notebook_vectors.append({
            "id": str(item["id"]),
            "notebook_name": item["title"],
            "notebook_content": item["content"],
            "notebook_name_vector": title_vector,
            "notebook_content_vector": content_vector
        })

    except Exception as e:
        print(f"Error embedding notebook ID {item['id']}: {e}")


In [ ]:

# Save to JSON for Azure Search Indexer
output_path = "notebook_index_data.json"
#os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(notebook_vectors, f, indent=2)

print("Azure Search document JSON created successfully.")


In [ ]:
from azure.search.documents import SearchClient
import json
from azure.core.credentials import AzureKeyCredential
credential = AzureKeyCredential(azure_search_api_key) 

# Upload some documents to the index
output_path = 'notebook_index_data.json'

with open(output_path, 'r') as file:  
    documents = json.load(file)  
search_client = SearchClient(endpoint=azure_search_endpoint, 
                             index_name="notebook-index", 
                             credential=credential)
result = search_client.upload_documents(documents)
print(f"Uploaded {len(documents)} documents") 